In [4]:
import os, requests
import numpy as np, pandas as pd
from astropy.io import fits

golden = pd.read_csv("paper_crossmatching/literature_x_legacysurvey_matches.csv")
RA, DEC, HP = "ra_mmu_ssl_legacysurvey_north", "dec_mmu_ssl_legacysurvey_north", "_healpix_29"

os.makedirs("golden_cutouts", exist_ok=True)
rows = []

for _, r in golden.iterrows():
    h = int(r[HP])
    path = f"golden_cutouts/{h}.npy"
    if os.path.exists(path):
        rows.append({"healpix": h, "ok": True}); continue

    url = ("https://www.legacysurvey.org/viewer/fits-cutout"
           f"?ra={r[RA]}&dec={r[DEC]}&layer=ls-dr9&pixscale=0.262&bands=grz&size=152")
    try:
        resp = requests.get(url, timeout=60)
    except Exception as e:
        print(h, "ERR", e, flush=True)
        rows.append({"healpix": h, "ok": False}); continue

    if resp.status_code != 200 or len(resp.content) < 10000:
        rows.append({"healpix": h, "ok": False}); continue

    tmp = f"golden_cutouts/{h}.fits"
    open(tmp, "wb").write(resp.content)
    with fits.open(tmp) as hd:
        flux = np.asarray(hd[0].data, dtype=np.float32)     # (3, 152, 152), g/r/z
    os.remove(tmp)

    np.save(path, {"flux": flux}, allow_pickle=True)
    rows.append({"healpix": h, "ok": True})
    print(h, flux.shape, flush=True)

cov = pd.DataFrame(rows)
cov.to_csv("golden_cutout_coverage.csv", index=False)
print(f"{cov.ok.sum()}/{len(golden)} fetched")

720429367133136635 (3, 152, 152)
315/315 fetched
